In [ ]:
from google.colab import drive
drive.mount('/content/drive')

The Google Drive is mounted, and the next steps involve processing CSV files from a specified directory.

In [ ]:
The data is sourced from a Google Drive folder. For this analysis, the `folder_path` is explicitly set in the following code cell.

In [ ]:
import os
import pandas as pd

# The identified folder path based on the file list
folder_path = '/content/drive/MyDrive/llm_benchmarking_data/'

# List all files in the specified folder
all_files = os.listdir(folder_path)


The files ending in `_01.csv` are filtered and loaded into a dictionary of pandas DataFrames.

In [ ]:
csv_01_files = [f for f in all_files if f.endswith('_01.csv')]

if not csv_01_files:
    print(f"No files ending with '_01.csv' found in '{folder_path}'. Please check the path or file names.")
else:
    dataframes = {}
    for file_name in csv_01_files:
        file_path = os.path.join(folder_path, file_name)
        # Use a cleaner name for the DataFrame key (e.g., 'A100_batch' from 'A100_batch_01.csv')
        df_name = file_name.replace('_01.csv', '')
        try:
            df = pd.read_csv(file_path)
            dataframes[df_name] = df
        except Exception as e:
            print(f"  - Error loading '{file_name}': {e}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os # Import os for directory creation and path manipulation

# Set plot style for IEEE readiness, ensuring consistency
plt.style.use('seaborn-v0_8-whitegrid') # A clean, professional style
plt.rcParams.update({
    'font.size': 12,          # Base font size
    'font.family': 'serif',   # Often preferred for academic papers
    'axes.labelsize': 14,     # Axis label font size
    'axes.titlesize': 16,     # Plot title font size
    'legend.fontsize': 10,    # Legend font size
    'xtick.labelsize': 10,    # X-axis tick label font size
    'ytick.labelsize': 10,    # Y-axis tick label font size
    'figure.titlesize': 18,   # Overall figure title (if used)
    'grid.linewidth': 0.5,     # Thinner grid lines
    'lines.linewidth': 1.5,    # Default line width
    'figure.autolayout': True, # Automatically adjust subplot params for tight layout
    'savefig.dpi': 300         # High DPI for saved figures
})

# Get the A100_sparce DataFrame
df_a100_sparce = dataframes['A100_sparce']

# Filter for the specific models requested, excluding Qwen and Naive Sparsity
models_to_compare_sparsity = [
    'Llama-3.1-8B (Dense FP16 Baseline)',
    'Llama-3.1-8B (MaskLLM 2:4)'
]

df_filtered_sparsity = df_a100_sparce[df_a100_sparce['Model'].isin(models_to_compare_sparsity)].copy()

# Simplify model names for better plot readability
df_filtered_sparsity['Simplified_Model'] = df_filtered_sparsity['Model'].replace({
    'Llama-3.1-8B (Dense FP16 Baseline)': 'Llama-3.1-8B (FP16)',
    'Llama-3.1-8B (MaskLLM 2:4)': 'Llama-3.1-8B (MaskLLM)'
})

# Define metrics to plot, excluding Perplexity
sparsity_metrics = {
    'Throughput': 'Throughput (tokens/s)',
    'Peak Memory (GB)': 'Peak Memory (GB)',
    'Avg Power (W)': 'Average Power (W)',
    'Total Energy (J)': 'Total Energy (J)'
}

# Create a multi-panel plot for sparsity comparison
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10)) # Adjust grid for 4 metrics
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, (col, ylabel) in enumerate(sparsity_metrics.items()):
    ax = axes[i]
    sns.barplot(x='Simplified_Model', y=col, data=df_filtered_sparsity, ax=ax, palette='mako', hue='Simplified_Model', legend=False, edgecolor='black')

    # Determine if lower or higher is better for title clarity
    direction_info = ''
    if col in ['Peak Memory (GB)', 'Avg Power (W)', 'Total Energy (J)']:
        direction_info = ' (Lower is Better)'
    elif col == 'Throughput':
        direction_info = ' (Higher is Better)'

    ax.set_title(f'{ylabel}{direction_info}', fontsize=plt.rcParams['axes.titlesize'] - 2) # Slightly smaller title for subplots
    ax.set_xlabel('') # Remove x-label, as we will rotate tick labels
    ax.set_ylabel(ylabel, fontsize=plt.rcParams['axes.labelsize'] - 2)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.tick_params(axis='x', rotation=30) # Rotate x-axis labels for readability

plt.suptitle('(A100) 2:4 Sparsity Data: FP16 vs MaskLLM', fontsize=plt.rcParams['figure.titlesize'])
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle

# --- Save the plot to a new directory ---

# Define the directory where results are stored in Google Drive (from earlier context)
RESULTS_DIR = "/content/drive/MyDrive/llm_benchmarking_data"

# Create a new subdirectory for plots
plots_dir = os.path.join(RESULTS_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

# Define the filename for the plot
plot_filename = "A100_sparsity_FP16_vs_MaskLLM.png"
plot_filepath = os.path.join(plots_dir, plot_filename)

# Save the figure
plt.savefig(plot_filepath)
plt.show()


### GPU Comparison for Llama-3.1-8B (FP16) Model

Let's compare the performance of the Llama-3.1-8B (FP16) model across different GPUs (A100, T4, L4) for various batch sizes (1, 8, 16). We will look at Throughput, Peak Memory, Average Power, and Total Energy.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# Prepare data for GPU comparison
gpu_comparison_data = []

for gpu_name_key, df in dataframes.items():
    if '_batch' in gpu_name_key:
        gpu_short_name = gpu_name_key.replace('_batch', '').upper() # e.g., 'A100'
        # Filter for all Llama-3.1-8B models, excluding MaskLLM (sparsity) and for relevant batch sizes
        df_filtered_gpu = df[
            df['Model'].str.startswith('Llama-3.1-8B') &
            (~df['Model'].str.contains('MaskLLM')) & # Exclude sparsity models
            (df['Batch Size'].isin([1, 8, 16]))
        ].copy()
        if not df_filtered_gpu.empty:
            df_filtered_gpu['GPU'] = gpu_short_name
            gpu_comparison_data.append(df_filtered_gpu)

if gpu_comparison_data:
    df_gpu_comparison = pd.concat(gpu_comparison_data, ignore_index=True)

    # Combine Model and GPU for the X-axis label to maintain grouped sorting
    df_gpu_comparison['Model_Short'] = df_gpu_comparison['Model'].str.replace('Llama-3.1-8B ', '', regex=False)
    df_gpu_comparison['Model_GPU'] = df_gpu_comparison['Model_Short'] + '\n(' + df_gpu_comparison['GPU'] + ')'

    # Define metrics to plot, swapping Throughput and Perplexity
    gpu_metrics = {
        'Throughput': 'Throughput (tokens/s)',
        'Perplexity': 'Perplexity',
        'Peak Memory (GB)': 'Peak Memory (GB)',
        'Avg Power (W)': 'Average Power (W)',
        'Total Energy (J)': 'Total Energy (J)'
    }

    # Find the boundaries and centers for the GPU segments
    unique_model_gpus = df_gpu_comparison['Model_GPU'].unique()
    gpu_boundaries = []
    gpu_spans = []
    current_gpu = None
    start_idx = 0

    for i, model_gpu in enumerate(unique_model_gpus):
        gpu = model_gpu.split('(')[-1].strip(')') # Extract GPU name
        if current_gpu is None:
            current_gpu = gpu
        elif current_gpu != gpu:
            gpu_boundaries.append(i - 0.5) # Place line exactly between categorical bars
            gpu_spans.append((current_gpu, start_idx, i - 1))
            current_gpu = gpu
            start_idx = i
    # Append the final GPU segment
    gpu_spans.append((current_gpu, start_idx, len(unique_model_gpus) - 1))

    # Create a single multi-panel plot for all comparisons
    fig_gpu, axes_gpu = plt.subplots(nrows=2, ncols=3, figsize=(22, 14))
    axes_gpu_flat = axes_gpu.flatten()

    # Variables to hold legend handles
    legend_handles, legend_labels = [], []

    for i, (col, ylabel) in enumerate(gpu_metrics.items()):
        ax = axes_gpu_flat[i]

        # Draw the plot. Allow legend on the first plot so we can grab the handles/colors
        draw_legend = (i == 0)
        sns.barplot(x='Model_GPU', y=col, hue='Batch Size', data=df_gpu_comparison, ax=ax, palette='viridis', edgecolor='black', legend=draw_legend)

        if i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            ax.get_legend().remove() # Remove from subplot so it doesn't overlap data

        # Determine if lower or higher is better for title clarity
        direction_info = ''
        if col in ['Perplexity', 'Peak Memory (GB)', 'Avg Power (W)', 'Total Energy (J)']:
            direction_info = ' (Lower is Better)'
        elif col == 'Throughput':
            direction_info = ' (Higher is Better)'

        ax.set_title(f'{ylabel}{direction_info}', fontsize=plt.rcParams['axes.titlesize'] - 2)
        ax.set_xlabel('') # We will handle x-labels manually below
        ax.set_ylabel(ylabel, fontsize=plt.rcParams['axes.labelsize'] - 2)
        ax.grid(axis='y', linestyle='--', alpha=0.7)

        # Clean up the tick labels to only show the model name (e.g., FP16)
        labels_text = [item.get_text().split('\n')[0].strip('()') for item in ax.get_xticklabels()]
        ax.set_xticks(ax.get_xticks()) # Fix the UserWarning by setting explicit ticks first
        ax.set_xticklabels(labels_text, rotation=45, ha='right')

        # Add vertical lines to separate GPUs
        for b in gpu_boundaries:
            ax.axvline(x=b, color='gray', linestyle='--', linewidth=1.5, alpha=0.8)

        # Add a single bold text label for each GPU segment inside the plot
        for gpu, start, end in gpu_spans:
            center = (start + end) / 2.0
            # Place the GPU label inside the plot at the top (y=0.95 in axes fraction)
            # bbox adds a slightly transparent white background to ensure text is visible over bars/grid
            ax.text(center, 0.95, gpu, ha='center', va='top', transform=ax.get_xaxis_transform(),
                    fontsize=14, fontweight='bold', color='black',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

    # Use the unused 6th subplot purely for the Legend
    ax_legend = axes_gpu_flat[5]
    ax_legend.axis('off') # Hide axes lines and ticks
    if legend_handles:
        ax_legend.legend(legend_handles, legend_labels, title='Batch Size', loc='center', fontsize=18, title_fontsize=20)

    plt.suptitle('Llama-3.1-8B Quantization Comparison Across GPUs and Batch Sizes', fontsize=plt.rcParams['figure.titlesize'] + 2)

    # Adjust layout to make space for suptitle. Since labels are moved inside, we don't need a large bottom margin anymore.
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.subplots_adjust(bottom=0.08, hspace=0.30)

    # Define the directory where results are stored in Google Drive (from earlier context)
    RESULTS_DIR = "/content/drive/MyDrive/llm_benchmarking_data"
    # Create a new subdirectory for plots if it doesn't exist
    plots_dir = os.path.join(RESULTS_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

    # Define the filename for the plot
    plot_filename_gpu = "Llama8B_Quantization_GPU_Comparison.png"
    plot_filepath_gpu = os.path.join(plots_dir, plot_filename_gpu)

    # Save the figure
    plt.savefig(plot_filepath_gpu, bbox_inches='tight')
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# Prepare data for small model (Qwen vs Llama 1B) comparison
small_model_data = []

for gpu_name_key, df in dataframes.items():
    if '_batch' in gpu_name_key:
        gpu_short_name = gpu_name_key.replace('_batch', '').upper() # e.g., 'A100'

        # Filter for Qwen and Llama 1B models, excluding sparsity/MaskLLM
        mask = (df['Model'].str.contains('Qwen', case=False) |
                (df['Model'].str.contains('Llama', case=False) & df['Model'].str.contains('1B', case=False)))

        df_filtered_small = df[mask & (~df['Model'].str.contains('MaskLLM', na=False))].copy()

        if not df_filtered_small.empty:
            # Find the largest batch size available for EACH model on this GPU
            idx = df_filtered_small.groupby('Model')['Batch Size'].idxmax()
            df_max_batch = df_filtered_small.loc[idx].copy()
            df_max_batch['GPU'] = gpu_short_name
            small_model_data.append(df_max_batch)

if small_model_data:
    df_small_comparison = pd.concat(small_model_data, ignore_index=True)

    # Clean up model names for a cleaner X-axis (e.g. removing 'Dense FP16 Baseline' verbosity)
    df_small_comparison['Model_Short'] = df_small_comparison['Model'] \
        .str.replace('Dense FP16 Baseline', 'FP16', regex=False) \
        .str.replace('1.5-1.8B', '1.8B', regex=False) \
        .str.replace('3.2-1B', '1B', regex=False)

    # Define the desired order for Model_Short values for consistent plotting
    model_short_order = [
        'Llama-1B (FP16)',
        'Llama-1B (INT8 bnb)',
        'Llama-1B (INT4 bnb)',
        'Qwen 1.8B (FP16)',
        'Qwen 1.8B (INT8 bnb)',
        'Qwen 1.8B (INT4 bnb)'
    ]

    # Convert 'Model_Short' to a categorical type with the defined order
    df_small_comparison['Model_Short'] = pd.Categorical(
        df_small_comparison['Model_Short'],
        categories=model_short_order,
        ordered=True
    )

    # Sort by GPU and then by the new categorical 'Model_Short'
    df_small_comparison = df_small_comparison.sort_values(by=['GPU', 'Model_Short'])

    # Re-create Model_GPU after sorting to reflect the new order
    df_small_comparison['Model_GPU'] = df_small_comparison['Model_Short'].astype(str) + '\n(' + df_small_comparison['GPU'] + ')'

    # Define custom palette to separate Llama (Blues) and Qwen (Oranges)
    llama_colors = sns.color_palette("Blues", 4)[1:] # Skip the lightest one to ensure visibility
    qwen_colors = sns.color_palette("Oranges", 4)[1:]

    custom_palette = {
        'Llama-1B (FP16)': llama_colors[2],      # Darkest blue
        'Llama-1B (INT8 bnb)': llama_colors[1],  # Medium blue
        'Llama-1B (INT4 bnb)': llama_colors[0],  # Light blue
        'Qwen 1.8B (FP16)': qwen_colors[2],      # Darkest orange
        'Qwen 1.8B (INT8 bnb)': qwen_colors[1],  # Medium orange
        'Qwen 1.8B (INT4 bnb)': qwen_colors[0]   # Light orange
    }

    # Define metrics to plot, keeping Throughput before Perplexity
    small_metrics = {
        'Throughput': 'Throughput (tokens/s)',
        'Perplexity': 'Perplexity',
        'Peak Memory (GB)': 'Peak Memory (GB)',
        'Avg Power (W)': 'Average Power (W)',
        'Total Energy (J)': 'Total Energy (J)'
    }

    # Find the boundaries and centers for the GPU segments
    unique_model_gpus = df_small_comparison['Model_GPU'].unique()
    gpu_boundaries = []
    gpu_spans = []
    current_gpu = None
    start_idx = 0

    for i, model_gpu in enumerate(unique_model_gpus):
        gpu = model_gpu.split('(')[-1].strip(')') # Extract GPU name
        if current_gpu is None:
            current_gpu = gpu
        elif current_gpu != gpu:
            gpu_boundaries.append(i - 0.5) # Place line exactly between categorical bars
            gpu_spans.append((current_gpu, start_idx, i - 1))
            current_gpu = gpu
            start_idx = i
    # Append the final GPU segment
    gpu_spans.append((current_gpu, start_idx, len(unique_model_gpus) - 1))

    # Create a single multi-panel plot
    fig_small, axes_small = plt.subplots(nrows=2, ncols=3, figsize=(22, 14))
    axes_small_flat = axes_small.flatten()

    # Variables to hold legend handles
    legend_handles, legend_labels = [], []

    for i, (col, ylabel) in enumerate(small_metrics.items()):
        ax = axes_small_flat[i]

        # Draw the plot. We'll use Model_Short as hue to give different colors to Qwen/Llama
        draw_legend = (i == 0)
        sns.barplot(x='Model_GPU', y=col, hue='Model_Short', data=df_small_comparison, ax=ax, palette=custom_palette, edgecolor='black', legend=draw_legend)

        if i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            ax.get_legend().remove() # Remove from subplot so it doesn't overlap data

        # Determine if lower or higher is better for title clarity
        direction_info = ''
        if col in ['Perplexity', 'Peak Memory (GB)', 'Avg Power (W)', 'Total Energy (J)']:
            direction_info = ' (Lower is Better)'
        elif col == 'Throughput':
            direction_info = ' (Higher is Better)'

        ax.set_title(f'{ylabel}{direction_info}', fontsize=plt.rcParams['axes.titlesize'] - 2)
        ax.set_xlabel('')
        ax.set_ylabel(ylabel, fontsize=plt.rcParams['axes.labelsize'] - 2)
        ax.grid(axis='y', linestyle='--', alpha=0.7)

        # Clean up the tick labels to only show the model name
        labels_text = [item.get_text().split('\n')[0].strip('()') for item in ax.get_xticklabels()]
        ax.set_xticks(ax.get_xticks())
        ax.set_xticklabels(labels_text, rotation=45, ha='right')

        # Add vertical lines to separate GPUs
        for b in gpu_boundaries:
            ax.axvline(x=b, color='gray', linestyle='--', linewidth=1.5, alpha=0.8)

        # Add a single bold text label for each GPU segment inside the plot
        for gpu, start, end in gpu_spans:
            center = (start + end) / 2.0
            # Place the GPU label inside the plot at the top
            ax.text(center, 0.95, gpu, ha='center', va='top', transform=ax.get_xaxis_transform(),
                    fontsize=14, fontweight='bold', color='black',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

    # Use the unused 6th subplot purely for the Legend
    ax_legend = axes_small_flat[5]
    ax_legend.axis('off')
    if legend_handles:
        ax_legend.legend(legend_handles, legend_labels, title='Model', loc='center', fontsize=18, title_fontsize=20)

    plt.suptitle('Qwen 1.8B vs Llama 1B Comparison (Batch Size: 16)', fontsize=plt.rcParams['figure.titlesize'] + 2)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.subplots_adjust(bottom=0.08, hspace=0.30)

    # Define the directory where results are stored in Google Drive
    RESULTS_DIR = "/content/drive/MyDrive/llm_benchmarking_data"
    # Create a new subdirectory for plots if it doesn't exist
    plots_dir = os.path.join(RESULTS_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

    # Define the filename for the plot
    plot_filename_small = "Qwen_Llama1B_Comparison.png"
    plot_filepath_small = os.path.join(plots_dir, plot_filename_small)

    # Save the figure
    plt.savefig(plot_filepath_small, bbox_inches='tight')
    plt.show()


### A100 Peak Memory Plot for Llama-3.1-8B Quantization Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# Filter data for A100 GPU
df_a100_peak_memory = df_gpu_comparison[df_gpu_comparison['GPU'] == 'A100'].copy()

# Define the specific metric for this plot
metric_col = 'Peak Memory (GB)'
metric_ylabel = 'Peak Memory (GB)'

# Set plot style for IEEE readiness, ensuring consistency
plt.style.use('seaborn-v0_8-whitegrid') # A clean, professional style
plt.rcParams.update({
    'font.size': 12,          # Base font size
    'font.family': 'serif',   # Often preferred for academic papers
    'axes.labelsize': 14,     # Axis label font size
    'axes.titlesize': 16,     # Plot title font size
    'legend.fontsize': 10,    # Legend font size
    'xtick.labelsize': 10,    # X-axis tick label font size
    'ytick.labelsize': 10,    # Y-axis tick label font size
    'figure.titlesize': 18,   # Overall figure title (if used)
    'grid.linewidth': 0.5,     # Thinner grid lines
    'lines.linewidth': 1.5,    # Default line width
    'figure.autolayout': True, # Automatically adjust subplot params for tight layout
    'savefig.dpi': 300         # High DPI for saved figures
})

fig_a100_peak, ax_a100_peak = plt.subplots(figsize=(10, 7)) # Adjust figure size for a single plot

sns.barplot(
    x='Model_GPU',
    y=metric_col,
    hue='Batch Size',
    data=df_a100_peak_memory,
    ax=ax_a100_peak,
    palette='viridis',
    edgecolor='black'
)

# Determine if lower or higher is better for title clarity
direction_info = ' (Lower is Better)'

ax_a100_peak.set_title(f'A100 {metric_ylabel}{direction_info}', fontsize=plt.rcParams['axes.titlesize'] + 2)
ax_a100_peak.set_xlabel('') # Remove x-label, as we will rotate tick labels
ax_a100_peak.set_ylabel(metric_ylabel, fontsize=plt.rcParams['axes.labelsize'] + 2)
ax_a100_peak.grid(axis='y', linestyle='--', alpha=0.7)

# Clean up the tick labels to only show the model name (e.g., FP16)
labels_text = [item.get_text().split('\n')[0].strip('()') for item in ax_a100_peak.get_xticklabels()]
ax_a100_peak.set_xticks(ax_a100_peak.get_xticks()) # Fix the UserWarning by setting explicit ticks first
ax_a100_peak.set_xticklabels(labels_text, rotation=45, ha='right')

# Add legend
ax_a100_peak.legend(title='Batch Size', loc='upper right', bbox_to_anchor=(1.25, 1))

plt.tight_layout()

# Save the plot to the plots directory
RESULTS_DIR = "/content/drive/MyDrive/llm_benchmarking_data"
plots_dir = os.path.join(RESULTS_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

plot_filename_a100_peak = "A100_Llama8B_Peak_Memory_Comparison.png"
plot_filepath_a100_peak = os.path.join(plots_dir, plot_filename_a100_peak)

plt.savefig(plot_filepath_a100_peak, bbox_inches='tight')
plt.show()


## Pareto Plots: Throughput vs. Peak Memory Usage

These plots show the trade-off between throughput and peak memory usage for Qwen 1.8B and Llama 1B models across different quantization levels (FP16, INT8, INT4) on each GPU. The points on the Pareto front represent configurations where you cannot improve one metric without sacrificing the other. Higher throughput and lower peak memory are generally desired.

In [ ]:
try:
    from adjustText import adjust_text
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "adjustText"])
    from adjustText import adjust_text

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# Define the directory where results are stored in Google Drive
RESULTS_DIR = "/content/drive/MyDrive/llm_benchmarking_data"
plots_dir = os.path.join(RESULTS_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

# 1. Prepare Llama-3.1-8B data for all batch sizes
llama_8b_pareto_data = []

for df_key, df_current in dataframes.items():
    df_filtered = df_current[
        df_current['Model'].str.contains('Llama-3.1-8B', case=False)
    ].copy()

    if not df_filtered.empty:
        # Extract GPU name from the dataframe key
        gpu_short_name = df_key.split('_')[0].upper()
        df_filtered['GPU'] = gpu_short_name
        llama_8b_pareto_data.append(df_filtered)

df_llama_8b_pareto = pd.concat(llama_8b_pareto_data, ignore_index=True)

# Simplify model names
df_llama_8b_pareto['Model_Short'] = df_llama_8b_pareto['Model'].copy()
df_llama_8b_pareto['Model_Short'] = df_llama_8b_pareto['Model_Short'].str.replace(r'Llama-3.1-8B \(Dense FP16 Baseline\)', 'FP16', regex=True)
df_llama_8b_pareto['Model_Short'] = df_llama_8b_pareto['Model_Short'].str.replace(r'Llama-3.1-8B \(FP16\)', 'FP16', regex=True)
df_llama_8b_pareto['Model_Short'] = df_llama_8b_pareto['Model_Short'].str.replace('Llama-3.1-8B ', '', regex=False)

# Remove Naive 2:4 Sparsity point
df_llama_8b_pareto = df_llama_8b_pareto[~df_llama_8b_pareto['Model_Short'].str.contains('Naive', case=False, na=False)]

# Define model order for consistency
model_short_order_pareto = [
    'FP16',
    '(INT8 bnb)',
    '(INT4 bnb)',
    '(AWQ INT4)',
    '(MaskLLM 2:4)'
]

def create_pareto_plot(df, gpu_name, metrics_x, metrics_y, title_x, title_y, plot_dir, model_order):
    df_gpu = df[df['GPU'] == gpu_name].copy()

    if df_gpu.empty:
        return

    # Sort data for Pareto front identification: minimize x, maximize y
    df_gpu_sorted = df_gpu.sort_values(by=[metrics_x, metrics_y], ascending=[True, False]).reset_index(drop=True)

    pareto_points = []
    max_y_seen = -float('inf')

    for idx, row in df_gpu_sorted.iterrows():
        current_x = row[metrics_x]
        current_y = row[metrics_y]

        if not pareto_points or current_y > max_y_seen:
            pareto_points.append(row)
            max_y_seen = current_y
        elif current_y == max_y_seen and current_x < pareto_points[-1][metrics_x]:
            # If same y, but better x, replace the last point
            pareto_points[-1] = row

    df_pareto = pd.DataFrame(pareto_points)

    # Increase figure width to give labels more room
    plt.figure(figsize=(14, 8))
    ax = sns.scatterplot(
        x=metrics_x,
        y=metrics_y,
        hue='Model_Short',
        size='Batch Size',
        sizes=(80, 250), # Slightly reduced point sizes to lower clutter
        data=df_gpu,
        palette='viridis',
        edgecolor='black',
        style='Model_Short',
        markers=True,
        hue_order=[m for m in model_order if m in df_gpu['Model_Short'].unique()]
    )

    # Plot Pareto front line
    if not df_pareto.empty:
        df_pareto_sorted_for_line = df_pareto.sort_values(by=metrics_x)
        plt.plot(
            df_pareto_sorted_for_line[metrics_x],
            df_pareto_sorted_for_line[metrics_y],
            color='red',
            linestyle='--',
            linewidth=2,
            label='Pareto Front'
        )

    plt.title(f'Pareto Front for {gpu_name}: Llama-3.1-8B\n{title_y} vs. {title_x}', fontsize=16)
    plt.xlabel(f'{title_x} (Lower is Better)', fontsize=14)
    plt.ylabel(f'{title_y} (Higher is Better)', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)

    # Clean up the legend (remove 'Model_Short' string, keep Batch Size)
    handles, labels = ax.get_legend_handles_labels()
    clean_handles = []
    clean_labels = []
    for h, l in zip(handles, labels):
        if l == 'Model_Short':
            continue
        clean_handles.append(h)
        clean_labels.append(l)

    # Place legend outside the plot area
    plt.legend(clean_handles, clean_labels, bbox_to_anchor=(1.02, 1), loc='upper left')

    # Label ONLY the most top left point (with override for A100)
    best_row = None
    if gpu_name == 'A100':
        override_df = df_gpu[(df_gpu['Model_Short'].str.contains('AWQ', na=False)) & (df_gpu['Batch Size'] == 16)]
        if not override_df.empty:
            best_row = override_df.iloc[0]

    if best_row is None:
        range_x = df_gpu[metrics_x].max() - df_gpu[metrics_x].min()
        range_y = df_gpu[metrics_y].max() - df_gpu[metrics_y].min()

        if range_x > 0 and range_y > 0:
            # Calculate normalized distance to the ideal top-left corner (0, 1)
            norm_x = (df_gpu[metrics_x] - df_gpu[metrics_x].min()) / range_x
            norm_y = (df_gpu[metrics_y] - df_gpu[metrics_y].min()) / range_y
            distances = np.sqrt((norm_x - 0)**2 + (norm_y - 1)**2)

            top_left_idx = distances.idxmin()
            best_row = df_gpu.loc[top_left_idx]

    if best_row is not None:
        label_text = best_row['Model_Short'].split('(')[0].strip() if '(' in best_row['Model_Short'] and 'bnb' not in best_row['Model_Short'] and 'AWQ' not in best_row['Model_Short'] else best_row['Model_Short']

        # Use ax.annotate to enforce a specific offset and prevent point overlap
        ax.annotate(
            f"Best Trade-off:\n{label_text}\n(bs: {best_row['Batch Size']})",
            xy=(best_row[metrics_x], best_row[metrics_y]),
            xytext=(50, -30), # Explicit offset: 50 points right, 30 points down
            textcoords='offset points',
            size=plt.rcParams['legend.fontsize'],
            color='black',
            bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1.0, alpha=0.95),
            arrowprops=dict(arrowstyle="-|>",

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# 1. Combine all available data into a single DataFrame
all_runs_data = []

for df_key, df in dataframes.items():
    temp_df = df.copy()
    # Extract GPU name from the dictionary key (e.g., 'A100_batch' -> 'A100')
    gpu_name = df_key.split('_')[0].upper()
    temp_df['GPU'] = gpu_name

    # Create a descriptive label for each run
    temp_df['Run_Config'] = temp_df['Model'] + ' | BS: ' + temp_df['Batch Size'].astype(str) + ' | ' + temp_df['GPU']

    all_runs_data.append(temp_df)

df_all_runs = pd.concat(all_runs_data, ignore_index=True)

# Filter for Llama-3.1-8B and exclude sparsity models
df_all_runs = df_all_runs[
    df_all_runs['Model'].str.contains('Llama-3.1-8B', case=False) &
    (~df_all_runs['Model'].str.contains('MaskLLM|Naive', case=False, na=False))
]

# 2. Filter out rows where Total Energy might be missing, then sort
df_all_runs = df_all_runs.dropna(subset=['Total Energy (J)'])
df_energy_sorted = df_all_runs.sort_values(by='Total Energy (J)', ascending=False).reset_index(drop=True)

# 3. Create the plot
# Adjusted figure size for vertical bars (wider rather than taller for a compact view)
plt.figure(figsize=(18, 10))

# Use a heatmap-like palette ('rocket') to represent energy consumption intuitively
sns.barplot(
    x='Run_Config',
    y='Total Energy (J)',
    data=df_energy_sorted,
    palette='rocket',
    hue='Run_Config',
    legend=False,
    edgecolor='black'
)

plt.title('Llama-3.1-8B Total Energy Consumption Ranking', fontsize=20, pad=20)
plt.xlabel('Configuration (Model | Batch Size | GPU)', fontsize=16)
plt.ylabel('Total Energy (Joules)', fontsize=16)

# Add gridlines for easier reading of values across the wide plot
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=12)

plt.tight_layout()

# Save the plot
energy_plot_filename = "Llama8B_Total_Energy_Ranking.png"
energy_plot_filepath = os.path.join(plots_dir, energy_plot_filename)
plt.savefig(energy_plot_filepath, bbox_inches='tight')
plt.show()


## Energy Efficiency: Tokens per Joule

This plot derives a new metric: **Energy Efficiency**, measured in Tokens per Joule. We calculate this by dividing the `Throughput (tokens/s)` by the `Average Power (W)`. Since 1 Watt = 1 Joule/second, this effectively gives us `(tokens/s) / (Joules/s) = tokens/Joule`. Higher values indicate a more energy-efficient and cost-effective generation process.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# 1. Combine all available data into a single DataFrame
efficiency_data = []

for df_key, df in dataframes.items():
    temp_df = df.copy()
    # Extract GPU name from the dictionary key
    gpu_name = df_key.split('_')[0].upper()
    temp_df['GPU'] = gpu_name

    # Create a descriptive label for each run
    temp_df['Run_Config'] = temp_df['Model'] + ' | BS: ' + temp_df['Batch Size'].astype(str) + ' | ' + temp_df['GPU']
    efficiency_data.append(temp_df)

df_eff = pd.concat(efficiency_data, ignore_index=True)

# Filter for Llama-3.1-8B and exclude sparsity models for a clean quantization comparison
# Added 'Sparsity' and 'Baseline' to catch all models from the sparsity datasets
df_eff = df_eff[
    df_eff['Model'].str.contains('Llama-3.1-8B', case=False) &
    (~df_eff['Model'].str.contains('MaskLLM|Naive|Sparsity|Baseline', case=False, na=False))
].copy()

# 2. Calculate the Efficiency Metric (Tokens per Joule)
# Drop rows where necessary metrics might be missing
df_eff = df_eff.dropna(subset=['Throughput', 'Avg Power (W)'])
df_eff['Efficiency (Tokens/Joule)'] = df_eff['Throughput'] / df_eff['Avg Power (W)']

# Sort from highest efficiency to lowest
df_eff_sorted = df_eff.sort_values(by='Efficiency (Tokens/Joule)', ascending=False).reset_index(drop=True)

# 3. Create the plot
plt.figure(figsize=(18, 10))

# Use a cool palette ('mako') to represent efficiency
sns.barplot(
    x='Run_Config',
    y='Efficiency (Tokens/Joule)',
    data=df_eff_sorted,
    palette='mako',
    hue='Run_Config',
    legend=False,
    edgecolor='black'
)

plt.title('Llama-3.1-8B Energy Efficiency Ranking (Tokens per Joule)\n(Higher is Better)', fontsize=20, pad=20)
plt.xlabel('Configuration (Model | Batch Size | GPU)', fontsize=16)
plt.ylabel('Efficiency (Tokens / Joule)', fontsize=16)

# Add gridlines for easier reading
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=12)

plt.tight_layout()

# Save the plot
eff_plot_filename = "Llama8B_Efficiency_Ranking.png"
eff_plot_filepath = os.path.join(plots_dir, eff_plot_filename)
plt.savefig(eff_plot_filepath, bbox_inches='tight')
plt.show()


## Pareto Plots: Energy Efficiency vs. Peak Memory Usage

These plots substitute Throughput for Energy Efficiency (Tokens per Joule, or tokens/sec/watt) on the Y-axis, while keeping Peak Memory on the X-axis. This reveals the trade-off between minimizing memory footprint and maximizing energy efficiency.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# Ensure the Efficiency column exists in the Pareto DataFrame
if 'Efficiency (Tokens/Joule)' not in df_llama_8b_pareto.columns:
    df_llama_8b_pareto['Efficiency (Tokens/Joule)'] = df_llama_8b_pareto['Throughput'] / df_llama_8b_pareto['Avg Power (W)']

def create_efficiency_pareto_plot(df, gpu_name, metrics_x, metrics_y, title_x, title_y, plot_dir, model_order):
    df_gpu = df[df['GPU'] == gpu_name].copy()

    # Drop rows that might have NaN for the required metrics
    df_gpu = df_gpu.dropna(subset=[metrics_x, metrics_y])

    if df_gpu.empty:
        return

    # Sort data for Pareto front identification: minimize x, maximize y
    df_gpu_sorted = df_gpu.sort_values(by=[metrics_x, metrics_y], ascending=[True, False]).reset_index(drop=True)

    pareto_points = []
    max_y_seen = -float('inf')

    for idx, row in df_gpu_sorted.iterrows():
        current_x = row[metrics_x]
        current_y = row[metrics_y]

        if not pareto_points or current_y > max_y_seen:
            pareto_points.append(row)
            max_y_seen = current_y
        elif current_y == max_y_seen and current_x < pareto_points[-1][metrics_x]:
            # If same y, but better x, replace the last point
            pareto_points[-1] = row

    df_pareto = pd.DataFrame(pareto_points)

    plt.figure(figsize=(14, 8))
    ax = sns.scatterplot(
        x=metrics_x,
        y=metrics_y,
        hue='Model_Short',
        size='Batch Size',
        sizes=(80, 250), # Slightly reduced point sizes to lower clutter
        data=df_gpu,
        palette='viridis',
        edgecolor='black',
        style='Model_Short',
        markers=True,
        hue_order=[m for m in model_order if m in df_gpu['Model_Short'].unique()]
    )

    # Plot Pareto front line
    if not df_pareto.empty:
        df_pareto_sorted_for_line = df_pareto.sort_values(by=metrics_x)
        plt.plot(
            df_pareto_sorted_for_line[metrics_x],
            df_pareto_sorted_for_line[metrics_y],
            color='red',
            linestyle='--',
            linewidth=2,
            label='Pareto Front'
        )

    plt.title(f'Pareto Front for {gpu_name}: Llama-3.1-8B\n{title_y} vs. {title_x}', fontsize=16)
    plt.xlabel(f'{title_x} (Lower is Better)', fontsize=14)
    plt.ylabel(f'{title_y} (Higher is Better)', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)

    # Clean up the legend (remove 'Model_Short' string, keep Batch Size)
    handles, labels = ax.get_legend_handles_labels()
    clean_handles = []
    clean_labels = []
    for h, l in zip(handles, labels):
        if l == 'Model_Short':
            continue
        clean_handles.append(h)
        clean_labels.append(l)

    # Place legend outside the plot area
    plt.legend(clean_handles, clean_labels, bbox_to_anchor=(1.02, 1), loc='upper left')

    # Label ONLY the most top left point (with override for A100)
    best_row = None
    if gpu_name == 'A100':
        override_df = df_gpu[(df_gpu['Model_Short'].str.contains('AWQ', na=False)) & (df_gpu['Batch Size'] == 16)]
        if not override_df.empty:
            best_row = override_df.iloc[0]

    if best_row is None:
        range_x = df_gpu[metrics_x].max() - df_gpu[metrics_x].min()
        range_y = df_gpu[metrics_y].max() - df_gpu[metrics_y].min()

        if range_x > 0 and range_y > 0:
            # Calculate normalized distance to the ideal top-left corner (0, 1)
            norm_x = (df_gpu[metrics_x] - df_gpu[metrics_x].min()) / range_x
            norm_y = (df_gpu[metrics_y] - df_gpu[metrics_y].min()) / range_y
            distances = np.sqrt((norm_x - 0)**2 + (norm_y - 1)**2)

            top_left_idx = distances.idxmin()
            best_row = df_gpu.loc[top_left_idx]

    if best_row is not None:
        label_text = best_row['Model_Short'].split('(')[0].strip() if '(' in best_row['Model_Short'] and 'bnb' not in best_row['Model_Short'] and 'AWQ' not in best_row['Model_Short'] else best_row['Model_Short']

        # Use ax.annotate to enforce a specific offset and prevent point overlap
        ax.annotate(
            f"Best Trade-off:\n{label_text}\n(bs: {best_row['Batch Size']})",
            xy=(best_row[metrics_x], best_row[metrics_y]),
            xytext=(50, -30), # Explicit offset
            textcoords='offset points',
            size=plt.rcParams['legend.fontsize'],
            color='black',
            bbox=dict(boxstyle="square,pad=0.4", fc="white", ec="black", lw=1.0, alpha=0.95),
            arrowprops=dict(arrowstyle="-|>",